<a href="https://colab.research.google.com/github/maryamfarhan/chaosbench-colab/blob/main/ChaosBench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!nproc
!python --version

Fri Nov 13 11:42:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/leap-stc/ChaosBench.git
%cd ChaosBench
!sed -E 's/==.*//' requirements.txt > req_unpinned.txt
!pip install -r req_unpinned.txt
!pip install zarr


Cloning into 'ChaosBench'...
remote: Enumerating objects: 1970, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (217/217), done.
remote: Total 1970 (delta 198), reused 311 (delta 132), pack-reused 1570 (from 1)
Receiving objects: 100% (1970/1970), 68.67 MiB | 17.25 MiB/s, done.
Resolving deltas: 100% (1060/1060), done.
/content/ChaosBench
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 9.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 14.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 48.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 51.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 33.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [lightning]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!df -h /content
%cd /content/ChaosBench
!mkdir -p data
%cd data
!wget https://huggingface.co/datasets/LEAP/ChaosBench/resolve/main/process.sh
!chmod +x process.sh

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  588M  113G   1% /
/content/ChaosBench
/content/ChaosBench/data
--2026-11-13 11:43:12--  https://huggingface.co/datasets/LEAP/ChaosBench/resolve/main/process.sh
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.23, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/LEAP/ChaosBench/9243dc670d85433b8c757c9608b21a52eb37ad44/process.sh?%2Fdatasets%2FLEAP%2FChaosBench%2Fresolve%2Fmain%2Fprocess.sh=&etag=%2253875ee5028d6f050584ad2bd6427987cff8791b%22 [following]
--2026-11-13 11:43:12--  https://huggingface.co/api/resolve-cache/datasets/LEAP/ChaosBench/9243dc670d85433b8c757c9608b21a52eb37ad44/process.sh?%2Fdatasets%2FLEAP%2FChaosBench%2Fresolve%2Fmain%2Fprocess.sh=&etag=%2253875ee5028d6f050584ad2bd6427987cff8791b%22
Reusing existing connection to huggingfa

In [4]:
%cd /content/ChaosBench/data
!apt-get install -y pigz > /dev/null 2>&1
!nohup ./process.sh era5 with_pigz > era5_log.txt 2>&1 &

/content/ChaosBench/data


In [ ]:
!tail -c 400 /content/ChaosBench/data/era5_log.txt
!df -h /content | tail -1
!ls -lh /content/ChaosBench/data

In [ ]:
%%writefile /content/ChaosBench/chaosbench/configs/unet_test_s2s.yaml
model_args:
    model_name: 'unet_s2s'
    input_size: 60
    output_size: 60
    learning_rate: 0.01
    num_workers: 2
    epochs: 1
    t_max: 1
    only_headline: False

data_args:
    batch_size: 4
    train_years: [1979, 1980, 1981]
    val_years: [2016]
    n_step: 1
    lead_time: 1
    land_vars: []
    ocean_vars: []

In [ ]:
!cd /content/ChaosBench/data && ./process.sh climatology with_pigz > climatology_log.txt 2>&1


In [ ]:
!ls /content/ChaosBench/data/era5 | wc -l
!du -sh /content/ChaosBench/data/era5
!ls /content/ChaosBench/data/climatology

In [ ]:
!cd /content/ChaosBench && python train.py --config_filepath chaosbench/configs/unet_test_s2s.yaml